# Classification-based Summarization


## Import packages


In [2]:
%load_ext autoreload
%autoreload 2
%env PYTORCH_ENABLE_MPS_FALLBACK=1
import torch
import numpy as np
from argsum import load_test_df, get_smatchtopr_classification_sums, get_barh_classification_sums

env: PYTORCH_ENABLE_MPS_FALLBACK=1


loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--distilbert-base-uncased/snapshots/12040accade4e8a0f71eabdb258fecc2e7e948be/config.json
Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "output_attentions": true,
  "output_hidden_states": true,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "transformers_version": "4.28.1",
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--distilbert-base-uncased/snapshots/12040accade4e8a0f71eabdb258fecc2e7e948be/vocab.txt
loading file added_tokens.json from cache at None
loading file spe

In [3]:
from time import time
from tqdm.notebook import tqdm
from itertools import product
import os
import json
import pandas as pd

###########################################################################################################################
### Get and evaluate classification based summaries ###
###########################################################################################################################


def get_classification_sums(
    df,
    get_classification_sums_callable,
    parameter_dict,
    output_dir="results/classification_sums",
    file_name=None,
):

    # Get unique topics and stances
    topics = df["topic"].unique().tolist()
    stances = [str(int(sta)) for sta in sorted(df["stance"].unique())]

    # Get parameter for iteration
    iterate_parameter_names = [
        item[0] for item in parameter_dict.items() if type(item[1]) == list
    ]
    iterate_parameter_values = [
        parameter_dict[parameter_name] for parameter_name in iterate_parameter_names
    ]
    iter_parameter_value_combinations = list(product(*iterate_parameter_values))

    # Create empty dict to store the clusters
    results = dict(
        zip(
            ["summaries", "parameter_names", "parameter_values"],
            [
                dict(
                    zip(
                        [str(comb) for comb in iter_parameter_value_combinations],
                        [
                            dict(
                                zip(
                                    topics,
                                    [
                                        dict(
                                            zip(
                                                stances,
                                                [
                                                    dict(
                                                        zip(
                                                            [
                                                                "sum_ids",
                                                                "sums",
                                                                "runtime",
                                                            ],
                                                            [None, None, None],
                                                        )
                                                    )
                                                    for i in range(len(stances))
                                                ],
                                            )
                                        )
                                        for i in range(len(topics))
                                    ],
                                )
                            )
                            for i in range(len(iter_parameter_value_combinations))
                        ],
                    )
                )
                for i in range(len(["summaries"]))
            ]
            + [iterate_parameter_names, iterate_parameter_values],
        )
    )

    ################################
    ### Iterate: topic & stance ####
    ################################

    for topic_stance in tqdm(
        [(topic, stance) for topic in topics for stance in stances],
        leave=True,
        desc="topic + stance",
    ):

        topic = topic_stance[0]
        stance = topic_stance[1]
        mask_topic_stance = (df["topic"] == topic) & (df["stance"] == int(stance))
        arguments = df[mask_topic_stance]["argument"].to_list()

        ############################
        ### Iterate: parameter #####
        ############################

        for comb in tqdm(
            iter_parameter_value_combinations,
            leave=False,
            desc="summarization parameter",
        ):
            iterate_parameter_dict = {
                **parameter_dict,
                **dict(zip(iterate_parameter_names, list(comb))),
            }

            ########################
            ### Get summaries ######
            ########################

            start_time = time()
            classification_sum_ids, classification_sums = (
                get_classification_sums_callable(
                    arguments, topic=topic, stance=int(stance), **iterate_parameter_dict
                )
            )
            runtime = time() - start_time

            if classification_sum_ids != None:
                results["summaries"][str(comb)][topic][stance]["sum_ids"] = [
                    int(id) for id in classification_sum_ids
                ]
                results["summaries"][str(comb)][topic][stance][
                    "sums"
                ] = classification_sums
                results["summaries"][str(comb)][topic][stance]["runtime"] = float(
                    np.round(runtime, 5)
                )

    ########################
    ### Save results #######
    ########################

    if file_name != None:
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        with open(output_dir + "/" + file_name, "w") as file:
            json.dump(results, file)

    return results

## Load data


In [4]:
ArgKP21 = load_test_df("ArgKP21")
Debate_test = load_test_df("Debate_test")

## BarH


### Extractive


In [6]:
barh_parameter_dict = {
    "quality_scorer_t": 0.7,
    "min_proportion_candidates": [0.1, 0.3],
    "match_scorer_t": [i for i in np.arange(0.75, 0.96, 0.025)],
    "final_match_scorer_t": 0,
    "use_llm": False,
}

barh_results = get_classification_sums(
    df=ArgKP21,
    get_classification_sums_callable=get_barh_classification_sums,
    parameter_dict=barh_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="ArgKP21_BarH.json",
)

topic + stance:   0%|          | 0/6 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f7235

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f7235

...

loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f7235

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f7235

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f7235

In [7]:
barh_parameter_dict = {
    "quality_scorer_t": 0.7,
    "min_proportion_candidates": [0.1, 0.3],
    "match_scorer_t": [i for i in np.arange(0.75, 0.96, 0.025)],
    "final_match_scorer_t": 0,
    "use_llm": False,
}

barh_results = get_classification_sums(
    df=Debate_test,
    get_classification_sums_callable=get_barh_classification_sums,
    parameter_dict=barh_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name=f"Debate_test_BarH.json",
)

topic + stance:   0%|          | 0/8 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f7235

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f7235

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f7235

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file config.json from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.28.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading file vocab.txt from cache at /Users/timaltendorf/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f7235

KeyboardInterrupt: 

### LLM


In [29]:
barh_key_points_parameter_dict = {
    "quality_scorer_t": 0.7,
    "min_proportion_candidates": 0,
    "match_scorer_t": [i for i in np.arange(0.75, 0.96, 0.025)],
    "final_match_scorer_t": 0,
    "use_llm": "candidates",
    "sum_token_length": 8,
    "sum_min_num": 12,
    "sum_min_num_plus": 8,
    "temperature": 0.5,
    "frequency_penalty": None,
    "few_shot": True,
}

barh_key_points_results = get_classification_sums(
    df=ArgKP21,
    get_classification_sums_callable=get_barh_classification_sums,
    parameter_dict=barh_key_points_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="ArgKP21_BarH_Candidates.json",
)

topic + stance:   0%|          | 0/6 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

models/transformers/roberta-large


  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 205, in start
    self.asyncio_loop.run_forever()
  File "/Users/timaltendorf/miniforge3/envs/argsum/lib/python3.10/asyncio/base_events.p

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'models/transformers/roberta-large'. Use `repo_type` argument if needed.

In [11]:
barh_key_points_parameter_dict = {
    "quality_scorer_t": 0.7,
    "min_proportion_candidates": 0,
    "match_scorer_t": [i for i in np.arange(0.75, 0.96, 0.025)],
    "final_match_scorer_t": 0,
    "use_llm": "candidates",
    "sum_token_length": 8,
    "sum_min_num": 12,
    "sum_min_num_plus": 8,
    "temperature": 0.5,
    "frequency_penalty": None,
    "few_shot": True,
}

barh_key_points_results = get_classification_sums(
    df=Debate_test,
    get_classification_sums_callable=get_barh_classification_sums,
    parameter_dict=barh_key_points_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="Debate_test_BarH_Candidates.json",
)

topic + stance:   0%|          | 0/8 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

In [12]:
barh_key_points_parameter_dict = {
    "quality_scorer_t": 0.7,
    "min_proportion_candidates": 0,
    "match_scorer_t": 0.8,
    "final_match_scorer_t": 0,
    "use_llm": "key_points",
    "sum_token_length": 8,
    "sum_min_num": [3, 4],
    "sum_min_num_plus": [2, 3, 4, 5, 6],
    "temperature": 0.5,
    "frequency_penalty": None,
    "few_shot": True,
}

barh_key_points_results = get_classification_sums(
    df=ArgKP21,
    get_classification_sums_callable=get_barh_classification_sums,
    parameter_dict=barh_key_points_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="ArgKP21_BarH_Key_Points.json",
)

topic + stance:   0%|          | 0/6 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

In [13]:
barh_key_points_parameter_dict = {
    "quality_scorer_t": 0.7,
    "min_proportion_candidates": 0,
    "match_scorer_t": 0.8,
    "final_match_scorer_t": 0,
    "use_llm": "key_points",
    "sum_token_length": 8,
    "sum_min_num": [3, 4],
    "sum_min_num_plus": [2, 3, 4, 5, 6],
    "temperature": 0.5,
    "frequency_penalty": None,
    "few_shot": True,
}

barh_key_points_results = get_classification_sums(
    df=Debate_test,
    get_classification_sums_callable=get_barh_classification_sums,
    parameter_dict=barh_key_points_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="Debate_test_BarH_Key_Points.json",
)

topic + stance:   0%|          | 0/8 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

## SMatchToPr


### Extractive


In [9]:
smatchtopr_parameter_dict = {
    "quality_scorer_t": 0.8,
    "min_proportion_candidates": [0.1, 0.3],
    "match_scorer_pr_t": 0.4,
    "damping_factor": 0.2,
    "final_match_scorer_t": 0,
    "scorer_cands": None,
    "scorer_cands_t": [i for i in np.arange(0.75, 0.96, 0.025)],
    "use_llm": False,
}

smatchtopr_results = get_classification_sums(
    df=ArgKP21,
    get_classification_sums_callable=get_smatchtopr_classification_sums,
    parameter_dict=smatchtopr_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="ArgKP21_SMatchToPr.json",
)

topic + stance:   0%|          | 0/6 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/config.json
Model config RobertaConfig {
  "_name_or_path": "models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/",
  "architectures": [
    "RobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.28.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50266
}

loading weights file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/pytorch_model.bin
All model checkpoint weights were used when initia

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/config.json
Model config RobertaConfig {
  "_name_or_path": "models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/",
  "architectures": [
    "RobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.28.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50266
}

loading weights file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/pytorch_model.bin
All model checkpoint weights were used when initia

...

loading configuration file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/config.json
Model config RobertaConfig {
  "_name_or_path": "models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/",
  "architectures": [
    "RobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.28.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50266
}

loading weights file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/pytorch_model.bin
All model checkpoint weights were used when initia

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/config.json
Model config RobertaConfig {
  "_name_or_path": "models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/",
  "architectures": [
    "RobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.28.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50266
}

loading weights file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/pytorch_model.bin
All model checkpoint weights were used when initia

summarization parameter:   0%|          | 0/18 [00:00<?, ?it/s]

loading configuration file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/config.json
Model config RobertaConfig {
  "_name_or_path": "models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/",
  "architectures": [
    "RobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.28.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50266
}

loading weights file models/match_scorer/bi_encoder/roberta_tp/2024-Feb-20_16-23-49/pytorch_model.bin
All model checkpoint weights were used when initia

In [ ]:
smatchtopr_parameter_dict = {
    "quality_scorer_t": [0.6, 0.8],
    "match_scorer_pr_t": 0.4,
    "damping_factor": 0.2,
    "final_match_scorer_t": 0,
    "scorer_cands": None,
    "scorer_cands_t": [i for i in np.arange(0.75, 0.96, 0.025)],
    "use_llm": False,
}

smatchtopr_results = get_classification_sums(
    df=Debate_test,
    get_classification_sums_callable=get_smatchtopr_classification_sums,
    parameter_dict=smatchtopr_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="Debate_test_SMatchToPr.json",
)

### LLM


In [22]:
smatchtopr_key_points_parameter_dict = {
    "quality_scorer_t": 0.8,
    "match_scorer_pr_t": 0.4,
    "damping_factor": 0.2,
    "final_match_scorer_t": 0,
    "scorer_cands_t": [i for i in np.arange(0.75, 0.96, 0.025)],
    "use_llm": "candidates",
    "sum_token_length": 8,
    "sum_min_num": 8,
    "sum_min_num_plus": 12,
    "temperature": 0.5,
    "frequency_penalty": None,
    "few_shot": True,
}

smatchtopr_key_points_results = get_classification_sums(
    df=ArgKP21,
    get_classification_sums_callable=get_smatchtopr_classification_sums,
    parameter_dict=smatchtopr_key_points_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="ArgKP21_SMatchToPr_Candidates.json",
)

topic + stance:   0%|          | 0/6 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

In [23]:
smatchtopr_key_points_parameter_dict = {
    "quality_scorer_t": 0.8,
    "match_scorer_pr_t": 0.4,
    "damping_factor": 0.2,
    "final_match_scorer_t": 0,
    "scorer_cands_t": [i for i in np.arange(0.75, 0.96, 0.025)],
    "use_llm": "candidates",
    "sum_token_length": 8,
    "sum_min_num": 8,
    "sum_min_num_plus": 12,
    "temperature": 0.5,
    "frequency_penalty": None,
    "few_shot": True,
}

smatchtopr_key_points_results = get_classification_sums(
    df=Debate_test,
    get_classification_sums_callable=get_smatchtopr_classification_sums,
    parameter_dict=smatchtopr_key_points_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="Debate_test_SMatchToPr_Candidates.json",
)

topic + stance:   0%|          | 0/8 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/9 [00:00<?, ?it/s]

In [24]:
smatchtopr_key_points_parameter_dict = {
    "quality_scorer_t": 0.8,
    "match_scorer_pr_t": 0.4,
    "damping_factor": 0.2,
    "final_match_scorer_t": 0,
    "use_llm": "key_points",
    "sum_token_length": 8,
    "sum_min_num": [3, 4],
    "sum_min_num_plus": [2, 3, 4, 5, 6],
    "temperature": 0.5,
    "frequency_penalty": None,
    "few_shot": True,
}

smatchtopr_key_points_results = get_classification_sums(
    df=ArgKP21,
    get_classification_sums_callable=get_smatchtopr_classification_sums,
    parameter_dict=smatchtopr_key_points_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="ArgKP21_SMatchToPr_Key_Points.json",
)

topic + stance:   0%|          | 0/6 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

5
6
7
8
9
6
7
8
9
10


summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

5
6
7
8
9
6
7
8
9
10


...

5
6
7
8
9
6
7
8
9
10


summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

5
6
7
8
9
6
7
8
9
10


summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

5
6
7
8
9
6
7
8
9
10


In [25]:
smatchtopr_key_points_parameter_dict = {
    "quality_scorer_t": 0.8,
    "match_scorer_pr_t": 0.4,
    "damping_factor": 0.2,
    "final_match_scorer_t": 0,
    "use_llm": "key_points",
    "sum_token_length": 8,
    "sum_min_num": [3, 4],
    "sum_min_num_plus": [2, 3, 4, 5, 6],
    "temperature": 0.5,
    "frequency_penalty": None,
    "few_shot": True,
}

smatchtopr_key_points_results = get_classification_sums(
    df=Debate_test,
    get_classification_sums_callable=get_smatchtopr_classification_sums,
    parameter_dict=smatchtopr_key_points_parameter_dict,
    output_dir="investigations/3_classification_summaries",
    file_name="Debate_test_SMatchToPr_Key_Points.json",
)

topic + stance:   0%|          | 0/8 [00:00<?, ?it/s]

summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

5
6
7
8
9
6
7
8
9
10


summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

5
6
7
8
9
6
7
8
9
10


...

5
6
7
8
9
6
7
8
9
10


summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

5
6
7
8
9
6
7
8
9
10


summarization parameter:   0%|          | 0/10 [00:00<?, ?it/s]

5
6
7
8
9
6
7
8
9
10
